In [5]:
import os
import json
import pathlib
from dataclasses import dataclass
from typing import Dict, List, Optional

# Force transformers to use only PyTorch (no TensorFlow / Keras backend)
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

PROJECT_ROOT = pathlib.Path('.').resolve()
PROC = PROJECT_ROOT / 'data_proc'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

In [6]:
class EmotionDataset(Dataset):
    """PyTorch dataset that returns input_ids, attention_mask, and labels."""
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_length: int = 128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=self.max_length,
        )
        item = {k: torch.tensor(v) for k, v in encoding.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

@dataclass
class EmotionConfig:
    model_name: str
    max_length: int = 128
    num_labels: int = 7

def load_splits() -> Dict[str, pd.DataFrame]:
    train_df = pd.read_csv(PROC / "train.csv")
    val_df = pd.read_csv(PROC / "val.csv")
    test_df = pd.read_csv(PROC / "test.csv")
    return {"train": train_df, "val": val_df, "test": test_df}

def load_label_mapping() -> Dict[int, str]:
    with open(PROC / "emotion_label_map.json", "r") as f:
        emo_id2name = json.load(f)
    emo_id2name = {int(k): v for k, v in emo_id2name.items()}
    return emo_id2name

def compute_metrics_builder(id2label: Dict[int, str]):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        acc = accuracy_score(labels, preds)
        macro_f1 = f1_score(labels, preds, average="macro")
        return {"accuracy": acc, "macro_f1": macro_f1}
    return compute_metrics

In [ ]:
def train_and_evaluate(config: EmotionConfig, output_name: str) -> None:
    emo_id2name = load_label_mapping()
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)

    splits = load_splits()
    train_df = splits["train"]
    val_df = splits["val"]
    test_df = splits["test"]

    text_col = "cleaned_utterance" if "cleaned_utterance" in train_df.columns else "utterance"

    train_dataset = EmotionDataset(
        texts=train_df[text_col].tolist(),
        labels=train_df["emotion_id"].tolist(),
        tokenizer=tokenizer,
        max_length=config.max_length,
    )
    val_dataset = EmotionDataset(
        texts=val_df[text_col].tolist(),
        labels=val_df["emotion_id"].tolist(),
        tokenizer=tokenizer,
        max_length=config.max_length,
    )
    test_dataset = EmotionDataset(
        texts=test_df[text_col].tolist(),
        labels=test_df["emotion_id"].tolist(),
        tokenizer=tokenizer,
        max_length=config.max_length,
    )

    id2label = {i: emo_id2name[i] for i in sorted(emo_id2name.keys())}
    label2id = {v: k for k, v in id2label.items()}

    model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name,
        num_labels=config.num_labels,
        id2label=id2label,
        label2id=label2id,
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    out_dir = MODELS_DIR / output_name

    # Use a minimal set of TrainingArguments options to avoid version incompatibilities
    training_args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_builder(id2label),
    )

    trainer.train()

    # Evaluate on test set
    preds_output = trainer.predict(test_dataset)
    logits = preds_output.predictions
    labels = preds_output.label_ids
    preds = np.argmax(logits, axis=-1)

    report = classification_report(
        labels,
        preds,
        target_names=[id2label[i] for i in sorted(id2label.keys())],
        digits=4,
    )

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")

    report_path = MODELS_DIR / f"{output_name}_evaluation_report.txt"
    with open(report_path, "w") as f:
        f.write(f"Model: {config.model_name} fine-tuned on balanced augmented dataset\n")
        f.write("=" * 72 + "\n\n")
        f.write(report)
        f.write("\n\n")
        f.write(f"Accuracy: {acc:.4f}\n")
        f.write(f"Macro F1: {macro_f1:.4f}\n")

    print(f"Saved evaluation report to: {report_path}")

In [8]:
# Choose model and run training/evaluation
model_name = "roberta-base"   # or "distilroberta-base"
config = EmotionConfig(model_name=model_name, max_length=128)
safe_name = model_name.replace("/", "-")
output_name = f"{safe_name}_emotion"
train_and_evaluate(config, output_name)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'